# JARVIS private model training (Google Colab)

This notebook fine-tunes an open base model with **your reviewed examples**. It does not train a foundation model from scratch. The free Colab GPU is not guaranteed. No paid compute is selected or required by this notebook.

**Privacy:** upload only examples you explicitly approved and sanitized. Do not include passwords, API keys, private messages, financial credentials, medical data, or other sensitive information. The notebook uploads the dataset and model to the Colab runtime for training. It does not publish the resulting model.

The starter dataset has only 20 examples and is for a pipeline smoke test, not a capable personal assistant. Add a larger, diverse, carefully reviewed dataset before expecting useful personalization.

Base model: `Qwen/Qwen3-1.7B` (Apache 2.0, text-only, chosen for compatibility with the current causal-LM training and llama.cpp export pipeline).

In [ ]:
import torch, os, subprocess, sys
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU (if available), then rerun.')
print(subprocess.check_output(['nvidia-smi'], text=True))

## 1. Get the current JARVIS training code

In [ ]:
%cd /content
!git clone --depth 1 --branch jarvis-offline-android-core https://github.com/opsayip96-byte/OpenJarvis.git
%cd /content/OpenJarvis
!pip -q install -r training/requirements.txt

## 2. Supply approved training examples

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select the JSONL exported from JARVIS Settings > Learning Lab.
if uploaded:
    dataset_path = '/content/' + next(iter(uploaded.keys()))
    print('Using uploaded dataset:', dataset_path)
else:
    dataset_path = '/content/OpenJarvis/training/data/jarvis_seed_sft.jsonl'
    print('Using the 20-example starter smoke-test dataset:', dataset_path)
print('Dataset bytes:', os.path.getsize(dataset_path))

## 3. Train a JARVIS LoRA adapter

The defaults are intentionally conservative for a free Colab GPU. If the runtime runs out of memory, reduce `max_length` in `training/train_lora.py` to 1024 and restart the runtime. Training may take a while and free sessions can disconnect.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3-1.7B'
OUTPUT_DIR = '/content/jarvis-lora'
!python training/train_lora.py --base-model "$BASE_MODEL" --train "$dataset_path" --output "$OUTPUT_DIR" --epochs 3 --batch-size 1 --grad-accum 8 --eval-fraction 0.1 --seed 42

## 4. Merge the adapter into a standalone Hugging Face model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
merged_dir = '/content/jarvis-merged'
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=False)
adapter = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = adapter.merge_and_unload()
merged.save_pretrained(merged_dir, safe_serialization=True, max_shard_size='2GB')
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=False)
tokenizer.save_pretrained(merged_dir)
print('Merged model saved to', merged_dir)

## 5. Convert to GGUF and quantize for Android

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
%cd /content/llama.cpp
!pip -q install -r requirements.txt gguf
!cmake -S . -B build -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF
!cmake --build build --target llama-quantize -j 2
!python convert_hf_to_gguf.py /content/jarvis-merged --outfile /content/jarvis-f16.gguf --outtype f16
!./build/bin/llama-quantize /content/jarvis-f16.gguf /content/JARVIS-Q4_K_M.gguf Q4_K_M
!ls -lh /content/JARVIS-Q4_K_M.gguf
!sha256sum /content/JARVIS-Q4_K_M.gguf

## 6. Download the JARVIS GGUF

Keep the SHA-256 output with the file. After download, import `JARVIS-Q4_K_M.gguf` in the Android app under Settings > Android Local Model > Import GGUF. The app's current importer and runtime still need a real-device compatibility test with this trained file.

In [ ]:
from google.colab import files
files.download('/content/JARVIS-Q4_K_M.gguf')